# Voice Layer — Usage Guide

A quick tour of the `voice/` package (`src/voice/`) for **Fatema** (RAG) and **Shahd** (UI).
This is a teaching notebook, not a test suite — see `tests/` for the real test suite and
`src/voice/CONTRACT.md` for the full frozen-contract reference.

**What it does:** Arabic (MSA) Speech-to-Text and Text-to-Speech, behind two functions:

```python
transcribe_audio(audio) -> TranscriptionResult   # speech -> MSA text
synthesize_speech(text) -> SynthesisResult       # MSA text -> spoken .wav
```

**Mock vs real:**
- **Mock mode (default — this whole notebook)** — zero setup: no models, no GPU, no API key.
  `transcribe_audio` returns a fixed sample transcript; `synthesize_speech` writes a short
  placeholder tone. Perfect for building the RAG/UI plumbing before real models are needed.
- **Real mode** — Whisper Large-v3 for STT, Azure Neural or offline Piper for TTS. Switched on
  with one env var (`VOICE_BACKEND=real`). Covered at the end of this notebook.

Every failure is an **exception** under `VoiceError` — you never get an error string back,
you always `try`/`except`. That's the one thing to remember.

In [1]:
import os, sys
sys.path.insert(0, "src")  # so `import voice` works from the repo root

# Force mock mode explicitly, before importing voice. voice/config.py loads a
# local .env if python-dotenv is installed; if a .env left over from real-mode
# testing sets VOICE_BACKEND=real, this notebook would silently try to run
# Whisper/Azure instead of mock. Setting it here first wins (dotenv defaults
# to never overriding an already-set env var), so this notebook is always
# guaranteed to run in mock mode regardless of your local .env/shell state.
os.environ["VOICE_BACKEND"] = "mock"

from voice import (
    transcribe_audio, transcribe_audio_async,
    synthesize_speech, synthesize_speech_async,
    VoiceError, AudioFormatError, TextValidationError,
    AZURE_VOICES,
)

print("voice package imported - running in mock mode by default, no setup needed.")

voice package imported - running in mock mode by default, no setup needed.


## 1. Speech → Text

`transcribe_audio` takes a file path, raw bytes, or a numpy waveform. In mock mode it doesn't
actually listen to the audio — it just validates the input and returns a fixed sample
transcript, so you can build against real-shaped data immediately.

In [2]:
# In real usage this would be a real recording (bytes from the browser, or a file path).
# In mock mode, any non-empty bytes works - the content is never inspected.
fake_audio = b"...pretend this is a recording from the UI..."

transcript = transcribe_audio(fake_audio)

print("text:    ", transcript.text)
print("backend: ", transcript.backend)   # "mock-stt" here; "whisper-large-v3" in real mode
print("language:", transcript.language)

text:     ما هي المهارات التي سأكتسبها عند دراسة تخصص الذكاء الاصطناعي وعلم البيانات؟
backend:  mock-stt
language: ar


## 2. Text → Speech

`synthesize_speech` takes MSA Arabic text and writes a `.wav` file. In mock mode it writes a
short placeholder tone (a real, valid, playable wav) instead of calling a TTS engine.

In [3]:
answer_text = "يتطلب تخصص الذكاء الاصطناعي وعلم البيانات إتمام مئة وستة وعشرين ساعة معتمدة للتخرج."

speech = synthesize_speech(answer_text)

print("audio_path:", speech.audio_path)
print("backend:   ", speech.backend)  # "mock-tts" here; "piper:ar_JO-kareem" / "azure:..." for real
print("duration:  ", speech.audio_duration_sec, "sec")

audio_path: C:\Users\sajaa\AppData\Local\Temp\tmp21q__nl1.wav
backend:    mock-tts
duration:   5.53 sec


## 3. Error handling — the one contract to remember

The voice layer **raises**, it never returns an error string. Every exception it can raise
inherits from `VoiceError`, so `except VoiceError` always catches a voice-layer failure if
you don't care about the specific reason. Catch a more specific subclass first if you want a
tailored message (e.g. "I didn't catch that" for silence vs. "couldn't read that file" for a
bad upload).

In [4]:
# A bad input type -> AudioFormatError (a subclass of VoiceError)
try:
    transcribe_audio(12345)  # not a path, bytes, or numpy array
except AudioFormatError:
    print("STT: bad input caught as AudioFormatError -", "ok")
except VoiceError:
    print("STT: caught as a generic VoiceError")

# Empty text -> TextValidationError (also a VoiceError)
try:
    synthesize_speech("")
except TextValidationError:
    print("TTS: empty text caught as TextValidationError -", "ok")
except VoiceError:
    print("TTS: caught as a generic VoiceError")

# This is the pattern to use in the real RAG/UI loop - catch the base class
# if you just want to know "did the voice layer fail?":
try:
    text = transcribe_audio(fake_audio).text
except VoiceError:
    text = None  # show a fallback message to the student instead of crashing

STT: bad input caught as AudioFormatError - ok
TTS: empty text caught as TextValidationError - ok


## 4. Async variants (for FastAPI)

Both functions have `_async` twins that offload the (potentially slow, real-mode) work to a
thread so an async event loop stays responsive. Same inputs, same return types, same
exceptions.

In [7]:
async def demo_async():
    transcript = await transcribe_audio_async(fake_audio)
    speech = await synthesize_speech_async(transcript.text)
    return transcript, speech

# In Jupyter, await the coroutine directly — the notebook already has a running loop
async_transcript, async_speech = await demo_async()
print("async STT backend:", async_transcript.backend)
print("async TTS backend:", async_speech.backend)

async STT backend: mock-stt
async TTS backend: mock-tts


## 5. The end-to-end loop

This is the shape of the real pipeline: audio in → transcribe → **Fatema's RAG answers the
question** → synthesize → audio out. The only thing that changes between mock and real mode
is which engine actually runs — this code never changes.

In [8]:
def fake_rag_answer(question: str) -> str:
    """PLACEHOLDER - this is where Fatema's real RAG pipeline plugs in.
    Takes the transcribed question, returns an MSA answer string."""
    return "يتطلب التخصص إتمام مئة وستة وعشرين ساعة معتمدة للتخرج."


def voice_loop(audio):
    try:
        question = transcribe_audio(audio).text
    except VoiceError:
        return None, "لم أفهم ما قلته، من فضلك حاول مرة أخرى."  # "I didn't catch that"

    # <<< SWAP THIS LINE for Fatema's real RAG call >>>
    answer_text = fake_rag_answer(question)

    try:
        speech = synthesize_speech(answer_text)
        return speech.audio_path, answer_text
    except VoiceError:
        return None, answer_text  # still show the text even if TTS failed


audio_path, answer_text = voice_loop(fake_audio)
print("answer text: ", answer_text)
print("answer audio:", audio_path)

answer text:  يتطلب التخصص إتمام مئة وستة وعشرين ساعة معتمدة للتخرج.
answer audio: C:\Users\sajaa\AppData\Local\Temp\tmparus70nv.wav


## 6. Switching to real mode + picking a voice

Real mode is one env var away — nothing about the calls above changes.

| Env var | Values | Purpose |
|---|---|---|
| `VOICE_BACKEND` | `mock` (default) / `real` | turns real Whisper + real TTS on |
| `VOICE_TTS` | `azure` / `piper` / `auto` (default) | which TTS engine, real mode only |
| `AZURE_SPEECH_KEY`, `AZURE_SPEECH_REGION` | your Azure key + region | needed for `azure`/`auto` |
| `PIPER_MODEL_PATH` (optional), `PIPER_LENGTH_SCALE` | local `.onnx` override / speaking rate | `piper`/`auto` |

`auto` (the real-mode default) tries Azure first and falls back to the fully-offline Piper
engine on any Azure failure, so real mode still works with no API key at all — just slower,
and needs `ffmpeg` and `piper` on PATH. Not sure your machine has what real mode needs?
`voice.health_check()` reports what's importable/configured, with no model loading or
network calls:

```python
from voice import health_check
health_check()
```

Turning real mode on (not run in this notebook — needs real credentials/models installed):

```python
import os
os.environ["VOICE_BACKEND"] = "real"
os.environ["VOICE_TTS"] = "auto"          # or "azure" / "piper"
os.environ["AZURE_SPEECH_KEY"] = "..."
os.environ["AZURE_SPEECH_REGION"] = "eastus"

transcript = transcribe_audio(real_audio_bytes)   # now runs Whisper Large-v3
speech = synthesize_speech(transcript.text)       # now runs Azure or Piper
```

**Picking a voice** works the same call in mock or real mode — `voice="male"`/`"female"` is
resolved via `AZURE_VOICES` and only affects the Azure engine (mock and Piper ignore it, but
happily accept the argument so your code doesn't need an `if` for it):

In [9]:
print("available named voices:", AZURE_VOICES)  # {"female": "...", "male": "..."}

male_voice = synthesize_speech("مرحباً", voice="male")
female_voice = synthesize_speech("مرحباً", voice="female")
print("male voice backend:  ", male_voice.backend)
print("female voice backend:", female_voice.backend)

available named voices: {'female': 'ar-SA-ZariyahNeural', 'male': 'ar-SA-HamedNeural'}
male voice backend:   mock-tts
female voice backend: mock-tts


## That's it

Two functions, one exception hierarchy, mock mode always available. For the full reference
(exact signatures, every exception type, the settled design decisions) see
`src/voice/CONTRACT.md`. For real bugs, open a GitHub issue against `voice/` with the failing
input attached — not Slack.